# EcoEye — Bird Image Scraper v2: eBird Source

**Iteration:** Switches the data source from iStock to the **eBird Macaulay Library** (`media.ebird.org`) for higher-quality, research-grade bird images.  
**Improvement over v1:** Filters images by file extension (`.jpg`, `.jpeg`, `.png`) rather than scraping all `<img>` tags blindly.

> ⚠️ This is a development iteration notebook. For the production scraper, see `Final Scraping (Code).ipynb`.

In [ ]:
from selenium import webdriver
from selenium.webdriver.chrome.service import Service
from selenium.webdriver.common.by import By
import requests
import os
import time

In [ ]:
# ─── CONFIGURATION ─────────────────────────────────────────────────────────────
# eBird species taxon code — find yours at https://ebird.org/explore
TAXON_CODE        = "afecuc1"          # African Emerald Cuckoo
CHROMEDRIVER_PATH = "chromedriver.exe" # Place chromedriver.exe in this folder, or provide full path
SPECIES_NAME      = "African_Emerald_Cuckoo"

URL = f"https://media.ebird.org/catalog?taxonCode={TAXON_CODE}&mediaType=photo"

SAVE_FOLDER = os.path.join(os.getcwd(), "downloaded_images", SPECIES_NAME)
os.makedirs(SAVE_FOLDER, exist_ok=True)
print(f"Saving images to: {SAVE_FOLDER}")

In [ ]:
# Initialize WebDriver
driver = webdriver.Chrome(service=Service(CHROMEDRIVER_PATH))
driver.get(URL)
time.sleep(5)  # Wait for dynamic content to load

# Filter images by extension
image_elements = driver.find_elements(By.TAG_NAME, 'img')
jpg_png_images = [
    img.get_attribute('src')
    for img in image_elements
    if img.get_attribute('src') and img.get_attribute('src').lower().endswith(('.png', '.jpg', '.jpeg'))
]

print(f"Found {len(jpg_png_images)} images")
driver.quit()

In [ ]:
# Download and save images
for image_url in jpg_png_images:
    response = requests.get(image_url)
    if response.status_code == 200:
        image_name = image_url.split('/')[-1]
        save_path = os.path.join(SAVE_FOLDER, image_name)
        with open(save_path, 'wb') as f:
            f.write(response.content)
        print(f"Downloaded: {image_name}")
    else:
        print(f"Failed ({response.status_code}): {image_url}")